# SatChecker independent validation

Validates Space-Track identifications against the IAU CPS SatChecker API.
Excludes SatChecker-merged rows so the check remains independent.

**Run from the repo root** (`hetdex_sats/`), not from `crossmatch/`.

Requires `crossmatch_and_make_catalog.ipynb` to have been run first
(needs `intermediate/HETDEX_PDR1_sats_matched.fits`).

In [3]:
import os, sys
import numpy as np
from astropy.table import Table
from astropy.io import fits

# Run from repo root; pipeline modules live in crossmatch/
sys.path.insert(0, os.path.join(os.path.abspath(".."), "crossmatch"))
if not os.path.basename(os.getcwd()) == "hetdex_sats":
    os.chdir("..")
    print(f"changed to {os.getcwd()}")

import satchecker_crosscheck as SC

CATALOG = "intermediate/HETDEX_PDR1_sats.fits"
OUT     = "intermediate/HETDEX_PDR1_sats_matched.fits"

assert os.path.exists(CATALOG), f"not found: {CATALOG}"
assert os.path.exists(OUT), f"not found: {OUT}"
print("ready")

ready


In [4]:
# Create a snapshot with SatChecker-merged rows excluded,
# so the validation is independent of the §5b merge.
SNAP = "intermediate/HETDEX_PDR1_sats_matched_stonly.fits"

m = Table.read(OUT, hdu="MATCH")
src = np.array([s.decode() if isinstance(s, bytes) else str(s)
                for s in m["id_source"]])
n = int((src == "satchecker").sum())
m["matched"][src == "satchecker"] = False

with fits.open(OUT) as hdul:
    h = fits.table_to_hdu(m); h.name = "MATCH"
    for k, hd in enumerate(hdul):
        if hd.name == "MATCH":
            hdul[k] = h
            break
    hdul.writeto(SNAP, overwrite=True)
print(f"excluded {n} satchecker rows; "
      f"{int(m['matched'].sum())} space-track matches to validate -> {SNAP}")

excluded 7 satchecker rows; 468 space-track matches to validate -> intermediate/HETDEX_PDR1_sats_matched_stonly.fits


In [5]:
# Run SatChecker validation against the space-track-only snapshot.
# ~2 s per streak; budget ~15 min for the full sample.
# Interrupt safely — partial results are written.
SC.main(["--catalog", CATALOG,
         "--matched", SNAP,
         "--n-sample", "999",
         "--out",     "crossmatch/satchecker_validation.csv",
         "--sleep",   "2.0"])

425 matched streaks eligible; 52 predate SatChecker's archive


[1/425] streak  169  space-track=  22969  satchecker=  22969  n_returned= 20  agree


[2/425] streak  172  space-track=  41790  satchecker=  41790  n_returned= 15  agree


[3/425] streak  123  space-track=  20762  satchecker=  20762  n_returned= 18  agree


[4/425] streak  199  space-track=  27950  satchecker=  27950  n_returned= 16  agree


...

[29/425] streak  155  space-track=  23342  satchecker=  23342  n_returned=  9  agree


[30/425] streak   84  space-track=  20623  satchecker=  20623  n_returned= 16  agree


[31/425] streak   73  space-track=  25875  satchecker=  25875  n_returned= 13  agree


[32/425] streak   55  space-track=   6797  satchecker=   6797  n_returned= 14  agree


[33/425] streak   97  space-track=  44421  satchecker=  44421  n_returned= 16  agree
